In [ ]:
import pandas as pd 
import numpy as np

In [ ]:
data = pd.read_csv("heart_disease_uci.csv")

In [ ]:
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)

In [ ]:
# One-hot encode the 'dataset' and 'cp' columns
data = pd.get_dummies(data, columns=['dataset', 'cp', 'restecg'])
data.head()

In [ ]:
# Perform mean imputation for the columns 'trestbps', 'chol', and 'fbs'
data['trestbps'].fillna(data['trestbps'].mean(), inplace=True)
data['chol'].fillna(data['chol'].mean(), inplace=True)

# Calculate the mode of the 'fbs' column
fbs_mode = data['fbs'].mode()[0]

# Impute missing values with the mode
data['fbs'].fillna(fbs_mode, inplace=True)




In [ ]:
# Check for missing values in the columns after imputation
print("Missing values in 'trestbps':", data['trestbps'].isnull().sum())
print("Missing values in 'chol':", data['chol'].isnull().sum())
print("Missing values in 'fbs':", data['fbs'].isnull().sum())


In [ ]:
import matplotlib.pyplot as plt

# List of columns to check for outliers
columns_to_check = ['age', 'trestbps', 'chol', 'thalch', 'num']

# Use descriptive statistics (IQR) to identify potential outliers
for column in columns_to_check:
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Find outliers
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]

    # Print outliers information
    print(f"Column: {column}")
    print(f"Number of outliers: {len(outliers)}")
    print(f"Outliers:\n{outliers[column].values}")
    print("-----------------------------------------------------")

# Visualize the data using boxplots for continuous columns
for column in columns_to_check:
    plt.figure(figsize=(10, 4))
    plt.boxplot(data[column].dropna(), vert=False)
    plt.title(f'Boxplot of {column}')
    plt.xlabel(column)
    plt.show()


In [ ]:
data.dtypes

In [ ]:


# List of boolean columns to convert to numerical binary
boolean_columns = [
    'fbs',
    'dataset_Cleveland', 
    'dataset_Hungary', 
    'dataset_Switzerland', 
    'dataset_VA Long Beach',
    'cp_asymptomatic', 
    'cp_atypical angina', 
    'cp_non-anginal', 
    'cp_typical angina',
    'restecg_lv hypertrophy',        
    'restecg_normal',
    'restecg_st-t abnormality'
    
]

# Convert the boolean columns to numerical (0 and 1)
data[boolean_columns] = data[boolean_columns].astype(int)

#Check 
data.dtypes


In [ ]:


# Convert 'sex' column to numeric: Female -> 0, Male -> 1
data['sex'] = data['sex'].map({'Female': 0, 'Male': 1})


In [ ]:


# Remove the 'slope', 'exang', and 'thal' columns from the dataset
data = data.drop(columns=['slope', 'exang', 'thal', 'oldpeak', 'ca'])



In [ ]:
data.shape

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

X = data.drop(columns=['id', 'num'])  # Predictors (all columns except 'id' and 'num')
y = data['num']  # Outcome variable

# Split the data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
xgboost_regressor = xgb.XGBRegressor(
    max_depth=4,           # Limits the maximum depth of a tree (prevents overfitting)
    learning_rate=0.1,     # Step size shrinkage used in updates to prevent overfitting
    n_estimators=100,      # Number of trees (more trees can increase accuracy but risk overfitting)
    subsample=0.8,         # Subsample ratio of the training instance to prevent overfitting
    colsample_bytree=0.8,  # Subsample ratio of columns when constructing each tree
    reg_alpha=0.01,        # L1 regularization term on weights (Lasso regression)
    reg_lambda=1,          # L2 regularization term on weights (Ridge regression)
    random_state=42        # Random state for reproducibility
)


In [ ]:
# Set up the initial XGBoost regressor

from sklearn.model_selection import GridSearchCV  # Import GridSearchCV
xgboost_regressor = xgb.XGBRegressor(random_state=42)

# Define the parameter grid for hyperparameter tuning
param_grid = {
    'max_depth': [3, 4, 5, 6],              # Different depths of trees
    'learning_rate': [0.01, 0.05, 0.1, 0.2], # Different learning rates
    'n_estimators': [50, 100, 150, 200],     # Different numbers of trees

}

# Set up GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=xgboost_regressor,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',  # Use negative MSE as the scoring metric for regression
    cv=5,                              # 5-fold cross-validation
    verbose=1,                         # Output progress
    n_jobs=-1                          # Use all available cores
)


In [ ]:
# Fit the model using GridSearchCV
grid_search.fit(X_train, y_train)



In [ ]:
#Get the best parameters and model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print(f"Best parameters: {best_params}")


In [ ]:
#Evaluate the best model on the test set
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

In [ ]:

train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print(f"Training MSE: {train_mse:.4f}, R2: {train_r2:.4f}")
print(f"Testing MSE: {test_mse:.4f}, R2: {test_r2:.4f}")

In [ ]:
# Classification task

# Recode the 'num' column: 0 stays as 0, 1, 2, 3, 4 are recoded to 1
data['num'] = data['num'].map({0: 0, 1: 1, 2: 1, 3: 1, 4: 1})

data['num'].value_counts()


In [ ]:
# Separate predictors (features) and the outcome variable
X = data.drop(columns=['id', 'num'])  # Use all columns except 'id' and 'num' as predictors
y = data['num']  # Outcome variable

# Split the data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
#Set up the initial XGBoost classifier
xgboost_classifier = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

# Define the parameter grid for hyperparameter tuning
param_grid = {
    'max_depth': [3, 4, 5, 6],              # Different depths of trees
    'learning_rate': [0.01, 0.05, 0.1, 0.2], # Different learning rates
    'n_estimators': [50, 100, 150, 200],     # Different numbers of trees
}

In [ ]:
#Set up GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=xgboost_classifier,
    param_grid=param_grid,
    scoring='accuracy',   # Use accuracy as the scoring metric for classification
    cv=5,                 # 5-fold cross-validation
    verbose=1,            # Output progress
    n_jobs=-1             # Use all available cores
)

# Fit the model using GridSearchCV
grid_search.fit(X_train, y_train)

# Get the best parameters and model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print(f"Best parameters: {best_params}")




In [ ]:
# Evaluate the best model on the training and test sets
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

# Evaluate model performance
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy: {test_accuracy:.4f}")

# Detailed evaluation metrics
print("\nClassification Report (Test Data):")
print(classification_report(y_test, y_pred_test))

print("\nConfusion Matrix (Test Data):")
print(confusion_matrix(y_test, y_pred_test))

In [ ]:
feature_importances = best_model.feature_importances_

# Create a DataFrame for better visualization
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

# Display the feature importance
print(feature_importance_df)

# Plot the Feature Importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='skyblue')
plt.xlabel('Feature Importance')
plt.title('Feature Importance in XGBoost Model')
plt.gca().invert_yaxis()  # To display the most important feature at the top
plt.show()
